In [ ]:
# Install libraries (अगर पहले से हैं तो skip हो जाएगा)
!pip install pandas numpy matplotlib seaborn scikit-learn

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ----------------------------
# Load Dataset
# ----------------------------

file_path = r"E:\project\instructor_effectiveness_dataset_2000_rows_-_Google_Sheets_1bdf6c39.csv"

df = pd.read_csv(file_path)

print("Dataset Preview:")
print(df.head())

print("\nDataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

# ----------------------------
# Data Cleaning
# ----------------------------

df.fillna(df.mean(numeric_only=True), inplace=True)

# ----------------------------
# Exploratory Data Analysis
# ----------------------------

plt.figure(figsize=(6,4))
sns.histplot(df["completion_rate"], kde=True)
plt.title("Completion Rate Distribution")
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

# ----------------------------
# Feature Engineering
# ----------------------------

df["engagement_score"] = (
    df["avg_watch_time"] +
    df["assignment_submission_rate"] +
    df["forum_activity_rate"]
) / 3

df["learning_score"] = (
    df["avg_score_improvement"] +
    df["avg_quiz_score"]
) / 2

df["feedback_quality"] = (
    df["avg_feedback_score"] *
    df["feedback_response_rate"]
)

# ----------------------------
# Instructor Effectiveness Score
# ----------------------------

df["effectiveness_score"] = (
    0.30 * df["completion_rate"] +
    0.20 * df["learning_score"] +
    0.20 * df["engagement_score"] +
    0.20 * df["feedback_quality"] -
    0.10 * df["dropout_rate"]
)

# ----------------------------
# Aggregate Instructor Data
# ----------------------------

instructor_df = df.groupby("instructor_id").agg({
    "completion_rate":"mean",
    "dropout_rate":"mean",
    "learning_score":"mean",
    "engagement_score":"mean",
    "feedback_quality":"mean",
    "effectiveness_score":"mean"
}).reset_index()

print("\nInstructor Data Preview:")
print(instructor_df.head())

# ----------------------------
# Create Effectiveness Tiers
# ----------------------------

instructor_df["tier"] = pd.qcut(
    instructor_df["effectiveness_score"],
    q=3,
    labels=["Low","Medium","High"]
)

print("\nTier Distribution:")
print(instructor_df["tier"].value_counts())

# ----------------------------
# Machine Learning Model
# ----------------------------

X = instructor_df.drop(["tier","instructor_id"], axis=1)
y = instructor_df["tier"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()

model.fit(X_train, y_train)

pred = model.predict(X_test)

# ----------------------------
# Evaluation
# ----------------------------

print("\nModel Accuracy:", accuracy_score(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred))

# Confusion Matrix
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# ----------------------------
# Feature Importance
# ----------------------------

importance = model.feature_importances_

feat = pd.Series(importance, index=X.columns)

plt.figure(figsize=(6,4))
feat.sort_values().plot(kind="barh")
plt.title("Feature Importance")
plt.show()

print("\nProject Completed Successfully 🚀")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Dataset Preview:
Empty DataFrame
Columns: [nan, nan, instructor_effectiveness_dataset_2000_rows.csv, nan, nan]
Index: []

Dataset Shape: (0, 5)


C:\Users\HP\AppData\Local\Temp\ipykernel_22100\3251186385.py:34: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")


KeyError: 'completion_rate'

<Figure size 600x400 with 0 Axes>